# Synthetic Dirichlet Dataset — Analysis

Reads `data/synthetic.csv` produced by `synthetic_dataset.py` and visualizes the 4-class Gaussian mixture with label shift over `t ∈ [0, 1]`.

Label key: `0 = C (+)`, `1 = D (+)`, `2 = A (−)`, `3 = B (−)`.

In [2]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("data/temporal_joint_shift-05.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/temporal_joint_shift-05.csv'

In [3]:
LABEL_NAMES = {0: "C (+)", 1: "D (+)", 2: "A (−)", 3: "B (−)"}
LABEL_COLORS = {"C (+)": "#D85A30", "D (+)": "#EF9F27", "A (−)": "#185FA5", "B (−)": "#1D9E75"}
df["class"] = df["label"].map(LABEL_NAMES)
df["t_frame"] = df["t"].round(3).astype(str)
sorted_frames = sorted(df["t_frame"].unique(), key=float)

pad = 0.3
fig = px.scatter(
    df,
    x="x1",
    y="x2",
    color="blob",
    color_discrete_map=LABEL_COLORS,
    animation_frame="t_frame",
    category_orders={"t_frame": sorted_frames, "class": list(LABEL_COLORS.keys())},
    opacity=0.6,
    range_x=[df["x1"].min() - pad, df["x1"].max() + pad],
    range_y=[df["x2"].min() - pad, df["x2"].max() + pad],
    title="Synthetic Dirichlet dataset — drift across t",
)
fig.update_traces(marker=dict(size=6))
fig.update_layout(width=800, height=600, legend_title_text="")
fig.layout.sliders[0].currentvalue = {"prefix": "t = "}
fig.show()

NameError: name 'df' is not defined

In [4]:
LABEL_NAMES = {0: "C (+)", 1: "D (+)", 2: "A (−)", 3: "B (−)"}
LABEL_COLORS = {"C (+)": "#D85A30", "D (+)": "#EF9F27", "A (−)": "#185FA5", "B (−)": "#1D9E75"}
df["class"] = df["label"].map(LABEL_NAMES)
df["t_frame"] = df["t"].round(3).astype(str)
sorted_frames = sorted(df["t_frame"].unique(), key=float)

pad = 0.3
fig = px.scatter(
    df,
    x="x1",
    y="x2",
    color="label",
    color_discrete_map=LABEL_COLORS,
    animation_frame="t_frame",
    category_orders={"t_frame": sorted_frames, "class": list(LABEL_COLORS.keys())},
    opacity=0.6,
    range_x=[df["x1"].min() - pad, df["x1"].max() + pad],
    range_y=[df["x2"].min() - pad, df["x2"].max() + pad],
    title="Synthetic Dirichlet dataset — drift across t",
)
fig.update_traces(marker=dict(size=6))
fig.update_layout(width=800, height=600, legend_title_text="")
fig.layout.sliders[0].currentvalue = {"prefix": "t = "}
fig.show()

NameError: name 'df' is not defined

# Compare generative approaches

Each section below visualizes one of the synthetic datasets produced by `python synthetic_dataset.py`. Blob colors are derived per dataset.

In [5]:
import pandas as pd
import plotly.express as px

BLOB_PALETTE = {
    "neg": "#185FA5",
    "A":   "#5BA1E0",
    "D":   "#EF9F27",
    "B":   "#1D9E75",
    "pos": "#D85A30",
}


def plot_dataset(csv_path, title, pad=0.3):
    df = pd.read_csv(csv_path)
    df["t_frame"] = df["t"].round(3).astype(str)
    sorted_frames = sorted(df["t_frame"].unique(), key=float)
    blob_order = [b for b in BLOB_PALETTE if b in set(df["blob"])]
    color_map = {b: BLOB_PALETTE[b] for b in blob_order}
    fig = px.scatter(
        df,
        x="x1",
        y="x2",
        color="blob",
        color_discrete_map=color_map,
        animation_frame="t_frame",
        category_orders={"t_frame": sorted_frames, "blob": blob_order},
        opacity=0.6,
        range_x=[df["x1"].min() - pad, df["x1"].max() + pad],
        range_y=[df["x2"].min() - pad, df["x2"].max() + pad],
        title=title,
    )
    fig.update_traces(marker=dict(size=6))
    fig.update_layout(width=800, height=600, legend_title_text="")
    fig.layout.sliders[0].currentvalue = {"prefix": "t = "}
    return fig

## Approach 0 — temporal_joint_shift (original)

Fixed blob geometry, prior shift between D and B. Blobs are static; only their relative counts move with `t`.

In [14]:
plot_dataset(
    "data/temporal_joint_shift.csv",
    "Approach 0 — temporal_joint_shift",
).show()

## Approach 1 — horizontal_shift

Two blobs (neg / pos), both means translated together along `+x1` as `t` grows (`SHIFT_RATE = 3.0`). Per-bag prevalence is balanced and smooth (`Dir(50·(0.5, 0.5))`). Pure spatial covariate shift: the classifier trained at `t = 0` learns an `x1` boundary, but both blobs slide past it.

In [17]:
plot_dataset(
    "data/horizontal_shift_labelshift_experiment/c5_s320.csv",
    "Approach 1 — horizontal_shift",
).show()

## Approach 2 — diagonal_shift

Same setup as approach 1, but both blobs translate along `(+x1, +x2)` simultaneously (`SHIFT_RATE = 3.0` on both axes). The trained classifier's boundary drifts off in two dimensions at once.

In [18]:
plot_dataset(
    "data/diagonal_shift_labelshift_experiment/c5_s320.csv",
    "Approach 2 — diagonal_shift",
).show()

## Approach 3 — vertical_shift

Same setup as approach 1, but the translation is along `+x2` instead of `+x1` (`SHIFT_RATE = 3.0`). Since the blobs are separated along `x1`, the classifier's `x1` boundary is preserved; this isolates a "perpendicular" form of covariate shift.

In [ ]:
plot_dataset(
    "data/vertical_shift_labelshift_experiment/c5_s640.csv",
    "Approach 3 — vertical_shift",
).show()

FileNotFoundError: [Errno 2] No such file or directory: 'data/vertical_shift_labelshift_experiment/c5_s640.csv'

## Approach 4 — spinning_shift

Two blobs rotate around their midpoint (`SPIN_TURNS = 2.0` full revolutions over `t ∈ [0, 1]`). At each half-turn the blobs fully swap sides — a classifier trained at `t = 0` is systematically wrong there — and they return to their starting positions at the end of each revolution.

In [20]:
plot_dataset(
    "data/temporal_joint_shift_spinning_single.csv",
    "Approach 4 — spinning_shift",
).show()

## Approach 5 — temporal_label_shift

Two stationary blobs (`neg` at `(-0.5, 0)`, `pos` at `(1.5, 0)`, isotropic `0.20 · I` covariances). Only the per-bag prevalence varies: the positive class grows smoothly from `0.3` to `0.7` across `t`, with each bag drawn from `Dir(50·(1−p, p))` so the proportion hugs the trajectory (std ≈ 0.07). Pure prior (label) shift — `P(X | Y)` is constant, only `P(Y)` moves.

In [ ]:
plot_dataset(
    "data/temporal_label_shift.csv",
    "Approach 5 — temporal_label_shift",
).show()

In [3]:
plot_dataset(
    "data/label_shift.csv",
    "Approach 6 — label_shift",
).show()

In [4]:
plot_dataset(
    "data/label_shift2.csv",
    "Approach 6 — label_shift2",
).show()

In [6]:
plot_dataset(
    "data/global_covariate_shift.csv",
    "Approach 7 — global_covariate_shift",
).show()

In [11]:
plot_dataset(
    "data/global_covariate_shift2.csv",
    "Approach 7 — global_covariate_shift2",
).show()

In [14]:
plot_dataset(
    "data/horizontal_mix.csv",
    "Approach 8 — horizontal_mix",
).show()